In [4]:
# Imports
from pathlib import Path
from experiment.utils import TrainedModel, TrainedModelID

import pandas as pd
import torch
from neuralhydrology.nh_run import start_run, eval_run, finetune
from experiment.eval import evaluate_models
import os
import yaml

In [28]:
model = TrainedModel(TrainedModelID.SOTA_20)

df = pd.read_csv(model.metrics_file, dtype={'basin':str})

basin_data = df.loc[df['NSE'] < df['NSE'].median()].sample(n=1)
basin = basin_data.basin.iloc[0]
nse = basin_data.NSE.iloc[0]
basin='08178880'


0.7929868400096893


'08178880'

In [29]:
# Add the path to the pre-trained model to the finetune config
file_path = "finetune.yml"

with open(file_path, "a") as fp:
    fp.write(f"\nbase_run_dir: {model.run_dir.absolute()}")

# Load the existing YAML data
with open(file_path, 'r') as f:
    data = yaml.safe_load(f)

data['experiment_name'] = f'basin_{basin}'  # Example modification

# Write back to the YAML file
with open(file_path, 'w') as f:
    yaml.dump(data, f)   


In [30]:
# Create a basin file with the basin we selected above
with open("finetune_basin.txt", "w") as fp:
    fp.write(basin)

In [31]:
# finetune
finetune(Path("finetune.yml"))

2024-09-18 19:08:39,392: Logging to /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08178880/output.log initialized.
2024-09-18 19:08:39,393: ### Folder structure created at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08178880
2024-09-18 19:08:39,393: ### Start finetuning with pretrained model stored in /home/admin/Fine-Flood-Forecasts/experiment/models/runs/sota_20
2024-09-18 19:08:39,394: ### Run configurations for basin_08178880
2024-09-18 19:08:39,394: additional_feature_files: None
2024-09-18 19:08:39,395: batch_size: 256
2024-09-18 19:08:39,395: checkpoint_path: None
2024-09-18 19:08:39,395: clip_gradient_norm: 1
2024-09-18 19:08:39,396: clip_targets_to_zero: ['QObs(mm/d)']
2024-09-18 19:08:39,397: commit_hash: 6dde7b4
2024-09-18 19:08:39,397: data_dir: /home/admin/Fine-Flood-Forecasts/data/CAMELS_US
2024-09-18 19:08:39,397: dataset: camels_us
2024-09-18 19:08:39,398: device: cuda:0
2024-09-18 19:08:39,398: dynamic_inp

/home/admin/Fine-Flood-Forecasts/neuralhydrology/training/basetrainer.py:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(str(checkpo

# Epoch 1: 100%|██████████| 13/13 [00:01<00:00,  9.47it/s, Loss: 0.0035]
2024-09-18 19:08:41,116: Epoch 1 average loss: avg_loss: 0.01718, avg_total_loss: 0.01718
2024-09-18 19:08:41,121: Setting learning rate to 5e-05
# Epoch 2: 100%|██████████| 13/13 [00:01<00:00,  9.26it/s, Loss: 0.2098]
2024-09-18 19:08:42,529: Epoch 2 average loss: avg_loss: 0.01716, avg_total_loss: 0.01716
# Epoch 3: 100%|██████████| 13/13 [00:01<00:00,  6.93it/s, Loss: 0.0004]
2024-09-18 19:08:44,412: Epoch 3 average loss: avg_loss: 0.01527, avg_total_loss: 0.01527
# Validation: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]
2024-09-18 19:08:45,470: Epoch 3 average validation loss: 0.03078 -- Median validation metrics: avg_loss: 0.03078, NSE: 0.49710, KGE: 0.30913, MSE: 1.22189
# Epoch 4: 100%|██████████| 13/13 [00:01<00:00,  9.19it/s, Loss: 0.0035]
2024-09-18 19:08:46,889: Epoch 4 average loss: avg_loss: 0.01904, avg_total_loss: 0.01904
# Epoch 5: 100%|██████████| 13/13 [00:01<00:00,  9.40it/s, Loss: 0.0005]
202

In [32]:
config_file_path = Path(os.path.abspath('')) / 'runs' / f'basin_{basin}' / 'config.yml'
finetuned_model = TrainedModel(config_file_path_or_experiment_name=config_file_path)
evaluate_models([model, finetuned_model], basins=[basin], include_benchmark=False, period='train')
evaluate_models([model, finetuned_model], basins=[basin], include_benchmark=False, period='validation')
evaluate_models([model, finetuned_model], basins=[basin], include_benchmark=False, period='test')

2024-09-18 19:09:00,156: Using the model weights from /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08178880/model_epoch010.pt


/home/admin/Fine-Flood-Forecasts/neuralhydrology/evaluation/tester.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(weight_file, m

# Evaluation: 100%|██████████| 1/1 [00:00<00:00,  2.95it/s]
2024-09-18 19:09:00,502: Stored metrics at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08178880/train/model_epoch010/train_metrics.csv
2024-09-18 19:09:00,503: Stored results at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08178880/train/model_epoch010/train_results.p


,Metric,sota_20,basin_08178880
0,NSE (mean),0.77,0.78
1,KGE (mean),0.46,0.57
2,MSE (mean),8.02,7.78
3,NSE (median),0.77,0.78
4,KGE (median),0.46,0.57
5,MSE (median),8.02,7.78


2024-09-18 19:09:00,553: Using the model weights from /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08178880/model_epoch010.pt


...

2024-09-18 19:09:01,410: Using the model weights from /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08178880/model_epoch010.pt


/home/admin/Fine-Flood-Forecasts/neuralhydrology/evaluation/tester.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(weight_file, m

# Evaluation: 100%|██████████| 1/1 [00:00<00:00,  2.86it/s]
2024-09-18 19:09:01,765: Stored metrics at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08178880/test/model_epoch010/test_metrics.csv
2024-09-18 19:09:01,766: Stored results at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08178880/test/model_epoch010/test_results.p


,Metric,sota_20,basin_08178880
0,NSE (mean),0.66,0.66
1,KGE (mean),0.72,0.72
2,MSE (mean),0.54,0.54
3,NSE (median),0.66,0.66
4,KGE (median),0.72,0.72
5,MSE (median),0.54,0.54


,Metric,sota_20,basin_08178880
0,NSE (mean),0.66,<b>0.66</b>
1,KGE (mean),0.72,<b>0.72</b>
2,MSE (mean),0.54,<b>0.54</b>
3,NSE (median),0.66,<b>0.66</b>
4,KGE (median),0.72,<b>0.72</b>
5,MSE (median),0.54,<b>0.54</b>
